# 🚕 Taxi Trip Duration — Solution (v3)

**Objectif :** Prédire la durée de trajet (en secondes) à partir des coordonnées GPS, métadonnées et features temporelles.

**Métriques d'évaluation :** MAE · RMSE · R²

---
### Structure du notebook
1. Imports & chargement des données  
2. Analyse exploratoire (EDA)  
3. Nettoyage & outliers  
4. Feature engineering — base  
5. Feature engineering — lag & rolling (sans fuite temporelle)  
6. Entraînement & comparaison de modèles  
7. Analyse des résultats & feature importance  
8. Conclusions & perspectives  


## 1. Imports & Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.facecolor'] = '#F7F9FC'
plt.rcParams['axes.facecolor']   = '#FAFBFD'
print('✅ Imports OK')

In [ ]:
# ── Chargement ─────────────────────────────────────────
# Remplacer le chemin par votre fichier réel
DATA_PATH = 'data.csv'   # 10k lignes (échantillon du train 875k)

df_raw = pd.read_csv(DATA_PATH, index_col=0)
print(f'Shape : {df_raw.shape}')
df_raw.head()

## 2. Analyse Exploratoire (EDA)

On examine la distribution de la cible, les valeurs manquantes et les corrélations entre variables.

In [ ]:
print('=== Types & valeurs manquantes ===')
print(df_raw.dtypes)
print('\nNaN :', df_raw.isnull().sum().sum())
print('\n=== Statistiques cible ===')
print(df_raw['trip_duration'].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution brute
axes[0].hist(df_raw['trip_duration'].clip(0, 5000), bins=50,
             color='#E74C3C', edgecolor='none', alpha=0.8)
axes[0].set_title('Distribution trip_duration (brut)')
axes[0].set_xlabel('Secondes')

# Vendor
df_raw.groupby('vendor_id')['trip_duration'].mean().plot(
    kind='bar', ax=axes[1], color=['#3498DB','#E74C3C'], edgecolor='none')
axes[1].set_title('Durée moyenne par vendor')
axes[1].set_xlabel('Vendor ID')

# Heure de la journée
df_raw['pickup_datetime'] = pd.to_datetime(df_raw['pickup_datetime'])
df_raw.groupby(df_raw['pickup_datetime'].dt.hour)['trip_duration'].mean().plot(
    kind='bar', ax=axes[2], color='#27AE60', edgecolor='none')
axes[2].set_title('Durée moyenne par heure')
axes[2].set_xlabel('Heure')

plt.tight_layout()
plt.show()

## 3. Nettoyage & Outliers

On supprime les trips aberrants :
- **< 60 s** : démarrages/arrêts immédiats, probablement des erreurs de capteur
- **> 7 200 s (2h)** : valeurs extrêmes (max observé = 86 337 s ≈ 24h)

Ces outliers représentent **0.6 %** des données et feraient exploser le RMSE.

In [ ]:
before = len(df_raw)
df = df_raw[(df_raw['trip_duration'] >= 60) & (df_raw['trip_duration'] <= 7200)].copy()
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df = df.sort_values('pickup_datetime').reset_index(drop=True)

print(f'Avant nettoyage : {before:,} lignes')
print(f'Après nettoyage : {len(df):,} lignes')
print(f'Supprimés       : {before - len(df)} ({(before-len(df))/before*100:.1f}%)')
print(f'\nNouvelle plage  : {df.trip_duration.min():.0f}s – {df.trip_duration.max():.0f}s')

## 4. Feature Engineering — Base

| Feature | Description |
|---------|-------------|
| `distance_km` | Distance Haversine pickup→dropoff (km) |
| `hour` | Heure de départ |
| `dayofweek` | Jour de la semaine (0=Lun … 6=Dim) |
| `month` | Mois |
| `is_weekend` | Flag week-end |
| `is_rush_hour` | Flag heure de pointe (7–9h, 17–19h) |
| `store_flag` | Encodage store_and_fwd_flag |
| `dist_x_hour` | Interaction distance × heure (congestion) |

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['distance_km']  = haversine(df.pickup_latitude, df.pickup_longitude,
                                df.dropoff_latitude, df.dropoff_longitude)
df['hour']         = df.pickup_datetime.dt.hour
df['dayofweek']    = df.pickup_datetime.dt.dayofweek
df['month']        = df.pickup_datetime.dt.month
df['is_weekend']   = (df.dayofweek >= 5).astype(int)
df['is_rush_hour'] = df.hour.isin([7,8,9,17,18,19]).astype(int)
df['store_flag']   = (df.store_and_fwd_flag == 'Y').astype(int)
df['dist_x_hour']  = df.distance_km * df.hour
df['hour_slot']    = df.hour // 3

print('Features de base créées.')
df[['distance_km','hour','dayofweek','is_weekend','is_rush_hour']].describe()

In [ ]:
# Heatmap de corrélation
BASE_COLS = ['vendor_id','passenger_count','distance_km','hour','dayofweek',
             'month','is_weekend','is_rush_hour','store_flag']
corr = df[BASE_COLS + ['trip_duration']].corr()
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Matrice de corrélation (features de base)')
plt.tight_layout()
plt.show()

## 5. Feature Engineering — Lag & Rolling

### Principe : zéro fuite temporelle
Chaque feature calculée pour la ligne `i` n'utilise que les lignes `0..i-1`,
grâce à un `.shift(1)` systématique avant tout `rolling()` ou `expanding()`.

### Features construites
| Feature | Type | Description |
|---------|------|-------------|
| `global_lag1/2` | Lag global | Durée des 2 trips précédents |
| `global_roll3_mean` | Rolling global | Moyenne glissante sur 3 trips |
| `global_roll5_std` | Rolling global | Écart-type sur 5 trips (volatilité trafic) |
| `global_roll10_mean` | Rolling global | Moyenne glissante sur 10 trips |
| `vendor_expand_mean/std` | Expanding par vendor | Historique cumulatif par vendeur |
| `hour_slot_mean` | Expanding par slot 3h | Proxy trafic par tranche horaire |
| `dow_mean` | Expanding par jour | Rythme hebdomadaire |
| `vendor_speed_mean` | Expanding par vendor | Vitesse moyenne historique (km/s) |

### ⚠️ Note importante sur cet échantillon
Ce dataset (10k/875k lignes) a un intervalle médian de **~16 min** entre deux trips consécutifs.
Sur le fichier complet, cet intervalle est **~20 secondes** → les lags globaux
capturent alors le niveau de trafic en quasi-temps réel.
Sur cet échantillon, seules les features **expanding par groupe** (`hour_slot_mean`,
`dow_mean`, `vendor_expand_mean`) apportent un signal réel.

In [ ]:
gm = df['trip_duration'].mean()   # pour cold-start (premières lignes)

# ── Lags globaux ─────────────────────────────────────────
s = df['trip_duration'].shift(1)
df['global_lag1']        = s.fillna(gm)
df['global_lag2']        = df['trip_duration'].shift(2).fillna(gm)
df['global_roll3_mean']  = s.rolling(3,  min_periods=1).mean().fillna(gm)
df['global_roll5_std']   = s.rolling(5,  min_periods=2).std().fillna(0)
df['global_roll10_mean'] = s.rolling(10, min_periods=3).mean().fillna(gm)

# ── Expanding par vendor ─────────────────────────────────
df['vendor_expand_mean'] = (df.groupby('vendor_id')['trip_duration']
                               .transform(lambda x: x.shift(1).expanding().mean())
                               .fillna(gm))
df['vendor_expand_std']  = (df.groupby('vendor_id')['trip_duration']
                               .transform(lambda x: x.shift(1).expanding().std())
                               .fillna(0))

# ── Expanding par tranche horaire (proxy trafic) ─────────
df['hour_slot_mean'] = (df.groupby('hour_slot')['trip_duration']
                           .transform(lambda x: x.shift(1).expanding().mean())
                           .fillna(gm))

# ── Expanding par jour de semaine ────────────────────────
df['dow_mean'] = (df.groupby('dayofweek')['trip_duration']
                    .transform(lambda x: x.shift(1).expanding().mean())
                    .fillna(gm))

# ── Vitesse moyenne par vendor ───────────────────────────
df['speed_proxy']      = df['distance_km'] / df['trip_duration'].replace(0, np.nan)
df['vendor_speed_mean'] = (df.groupby('vendor_id')['speed_proxy']
                              .transform(lambda x: x.shift(1).expanding().mean())
                              .fillna(df['speed_proxy'].mean()))

LAG_COLS = ['global_lag1','global_lag2','global_roll3_mean','global_roll5_std',
            'global_roll10_mean','vendor_expand_mean','vendor_expand_std',
            'hour_slot_mean','dow_mean','vendor_speed_mean']

print(f'Lag/rolling features créées : {len(LAG_COLS)}')
print(f'NaN restants : {df[LAG_COLS].isnull().sum().sum()}')

In [ ]:
# Corrélations des lag features avec la cible
lag_corrs = {f: df[f].corr(df['trip_duration']) for f in LAG_COLS}
lag_corrs_s = dict(sorted(lag_corrs.items(), key=lambda x: abs(x[1])))

colors = ['#E74C3C' if abs(v) > 0.04 else '#BDC3C7' for v in lag_corrs_s.values()]
plt.figure(figsize=(8, 4))
plt.barh(list(lag_corrs_s.keys()), list(lag_corrs_s.values()), color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.axvline(0.04,  color='#E74C3C', lw=1, linestyle='--', alpha=0.6, label='seuil ±0.04')
plt.axvline(-0.04, color='#E74C3C', lw=1, linestyle='--', alpha=0.6)
plt.title('Corrélation lag/rolling features vs trip_duration\n'
          '(rouge = signal utile | gris = bruit sur échantillon)')
plt.xlabel('Corrélation de Pearson')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Entraînement & Comparaison de Modèles

### Justification des métriques
| Métrique | Justification |
|----------|---------------|
| **MAE** | Directement interprétable en secondes ; robuste aux outliers |
| **RMSE** | Pénalise davantage les grandes erreurs (un taxi très en retard est plus coûteux) |
| **R²** | Mesure la proportion de variance expliquée ; comparable entre modèles |

### Split chronologique
On utilise un split **80/20 chronologique** (pas de mélange aléatoire),
ce qui reflète le déploiement réel : le modèle est entraîné sur le passé
et prédit le futur. Un split aléatoire introduirait une fuite temporelle.

In [ ]:
BASE_COLS = ['vendor_id','passenger_count','distance_km','hour','dayofweek',
             'month','is_weekend','is_rush_hour','store_flag','dist_x_hour']
ALL_COLS  = BASE_COLS + LAG_COLS

split = int(len(df) * 0.8)
X_base_tr, X_base_te = df[BASE_COLS].iloc[:split], df[BASE_COLS].iloc[split:]
X_all_tr,  X_all_te  = df[ALL_COLS].iloc[:split],  df[ALL_COLS].iloc[split:]
y_tr, y_te = df['trip_duration'].iloc[:split], df['trip_duration'].iloc[split:]

print(f'Train : {len(X_all_tr):,} lignes  |  {df.pickup_datetime.iloc[0].date()} → {df.pickup_datetime.iloc[split-1].date()}')
print(f'Test  : {len(X_all_te):,} lignes  |  {df.pickup_datetime.iloc[split].date()} → {df.pickup_datetime.iloc[-1].date()}')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def evaluate(name, model, X_tr, y_tr, X_te, y_te, transform=None):
    y_fit = np.sqrt(y_tr) if transform == 'sqrt' else y_tr
    model.fit(X_tr, y_fit)
    p = model.predict(X_te)
    if transform == 'sqrt': p = np.clip(p, 0, None) ** 2
    r = dict(name=name, preds=p,
             MAE=mean_absolute_error(y_te,p),
             RMSE=np.sqrt(mean_squared_error(y_te,p)),
             R2=r2_score(y_te,p))
    print(f"  {name:38s}  MAE={r['MAE']:>5.0f}s  RMSE={r['RMSE']:>5.0f}s  R²={r['R2']:.4f}")
    return r

print('── Baseline (sans lag/rolling) ──────────────────────────────')
results = [
    evaluate('Ridge — base',
             Pipeline([('sc',StandardScaler()),('m',Ridge())]),
             X_base_tr, y_tr, X_base_te, y_te),
    evaluate('Random Forest — base',
             RandomForestRegressor(n_estimators=300,max_depth=10,
                                   min_samples_leaf=3,random_state=42,n_jobs=-1),
             X_base_tr, y_tr, X_base_te, y_te),
    evaluate('GBM — base',
             GradientBoostingRegressor(n_estimators=300,max_depth=4,
                                       learning_rate=0.05,subsample=0.8,random_state=42),
             X_base_tr, y_tr, X_base_te, y_te),
]

print('── v3 (base + lag/rolling) ──────────────────────────────────')
results += [
    evaluate('Ridge — base+lag',
             Pipeline([('sc',StandardScaler()),('m',Ridge())]),
             X_all_tr, y_tr, X_all_te, y_te),
    evaluate('GBM — base+lag (raw target)',
             GradientBoostingRegressor(n_estimators=300,max_depth=4,
                                       learning_rate=0.05,subsample=0.8,random_state=42),
             X_all_tr, y_tr, X_all_te, y_te),
    evaluate('GBM — base+lag (√target)',
             GradientBoostingRegressor(n_estimators=300,max_depth=4,
                                       learning_rate=0.05,subsample=0.8,random_state=42),
             X_all_tr, y_tr, X_all_te, y_te, transform='sqrt'),
]

## 7. Analyse des Résultats & Feature Importance

In [ ]:
best = min(results, key=lambda r: r['RMSE'])
print(f"\n✅ Meilleur modèle : {best['name']}")
print(f"   MAE  = {best['MAE']:.0f}s  (~{best['MAE']/60:.1f} min)")
print(f"   RMSE = {best['RMSE']:.0f}s  (~{best['RMSE']/60:.1f} min)")
print(f"   R²   = {best['R2']:.4f}")

# Tableau récap
summary = pd.DataFrame([{k:v for k,v in r.items() if k not in ['model','preds']}
                         for r in results])
summary['MAE_min']  = (summary['MAE'] / 60).round(1)
summary['RMSE_min'] = (summary['RMSE'] / 60).round(1)
summary[['name','MAE','RMSE','R2','MAE_min','RMSE_min']]

In [ ]:
# Feature importance du meilleur GBM avec lag
gbm_lag = GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                     learning_rate=0.05, subsample=0.8, random_state=42)
gbm_lag.fit(X_all_tr, y_tr)
feat_imp = pd.Series(gbm_lag.feature_importances_, index=ALL_COLS).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top 15 features
top = feat_imp.head(15)
colors_fi = ['#E74C3C' if f in LAG_COLS else '#3498DB' for f in top.index[::-1]]
axes[0].barh(top.index[::-1], top.values[::-1], color=colors_fi, edgecolor='none')
axes[0].set_title('Feature Importance — GBM (base+lag)\n🔴 lag/rolling  🔵 base',
                   fontweight='bold')
axes[0].set_xlabel('Importance')

# Actual vs predicted
axes[1].scatter(y_te, best['preds'], alpha=0.25, s=8, c='#E74C3C', edgecolors='none')
lim = [0, max(y_te.max(), best['preds'].max())]
axes[1].plot(lim, lim, 'k--', lw=1.2, label='Parfait')
axes[1].set_title(f"Réel vs Prédit ({best['name']})", fontweight='bold')
axes[1].set_xlabel('Réel (s)'); axes[1].set_ylabel('Prédit (s)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution des résidus
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

residuals = y_te.values - best['preds']
axes[0].hist(residuals, bins=40, color='#9B59B6', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='black', lw=1.5, linestyle='--')
axes[0].axvline(np.median(residuals), color='#E74C3C', lw=1.5,
                linestyle='--', label=f'médiane={np.median(residuals):.0f}s')
axes[0].set_title('Distribution des résidus'); axes[0].set_xlabel('Résidu (s)')
axes[0].legend()

# Résidus vs distance
axes[1].scatter(df['distance_km'].iloc[split:], residuals,
                alpha=0.2, s=8, c='#3498DB', edgecolors='none')
axes[1].axhline(0, color='black', lw=1, linestyle='--')
axes[1].set_title('Résidus vs Distance')
axes[1].set_xlabel('Distance (km)'); axes[1].set_ylabel('Résidu (s)')

plt.tight_layout()
plt.show()

## 8. Conclusions & Perspectives

### Résultats obtenus (split chronologique 80/20)

| Modèle | MAE | RMSE | R² |
|--------|-----|------|----|
| Ridge — base | ~308s | ~434s | 0.615 |
| **GBM — base** | **~257s** | **~379s** | **0.707** |
| GBM — base+lag (√target) | ~255s | ~383s | 0.700 |